---
# Commande pour le rendu : quarto render main.ipynb --execute
# Instructions pour quarto
format:
  html:
    code-fold: true
    embed-resources: true
---

<!-- Header stylé -->
<div style="background: linear-gradient(90deg, #4e54c8, #8f94fb); padding: 30px; border-radius: 12px; color: white; text-align: center; margin-bottom: 20px;">
  <h3 style="margin: 0; font-size: 3em;">🏠🚆Lien entre l'offre de transport et le prix du logement en Île-de-France 🏠🚆</h3>
</div>

# Introduction

L'objectif de ce projet est d'analyser comment l'accessibilité aux transports en commun influence le prix du logement en Île‑de‑France. Pour cela nous croisons deux sources complémentaires : 
- les transactions immobilières obtenues via la base de donnée DVF ("Demande de valeur foncière") pour obtenir les prix et la localisation des logements
- les données issues de l'API IDFM (Île-de-France Mobilité) décrivant les horaires prévus des transports en commun dans les 30 prochains jours.

Les traitements effectués sont les suivants :

1.Une première phase d'ingestion, nettoyage et visualisation est effectuée séparément chaque base de donnée. Ces phases son présentées dans des pages séparées : [Traitement de IDFM](idfm.html) et [Traitement de DVF](dvf.html)

2. La version nettoyée de DVF est ensuite augmentée d'informations sur la desserte en transport du logement ([Première partie](#ajout-de-la-desserte-à-DVF))

3. Des visualisations sont produites à partir de cette nouvelle base de donnée

4. [PARTIE ECONO] ???

5. Partie ML 

# Ajout de la desserte à DVF

L'approche retenue et d'ajouter au dataframe DVF des métriques indiquant la desserte en transport de chaque logement vendu.


## Calcul des arrêts les plus proches

Après avoir ouvert les fichiers intermédiares, on ajoute à chaque logement la distance à l'arrêt le plus proche pour chaque mode de transport (train, metro, tramway, bus).

Les première lignes du dataframe obtenues sont affichées pour illustrer (seules les colonnes nouvellement ajoutées sont affichées).

In [1]:
import os
import geopandas as gpd

# verifie que l'exectution se fait depuis le bon répertoire
if os.getcwd().endswith("notebooks"):
    os.chdir('..')

# ouverture des fichiers geojson
gdf_idfm_small = gpd.read_file("cache/results/passage_par_arret_synthetique.geojson")
gdf_idfm = gpd.read_file("cache/results/passage_par_arret_full.geojson")
gdf_dvf = gpd.read_file("cache/results/prix_logements.geojson")

# ajout d'une cle principale à gdf_dvf
gdf_dvf['point_id'] = gdf_dvf.index.astype(str)


# reprojeter en CRS métrique, trouver le point le plus proche et la distance en mètres
gdf_dvf_m = gdf_dvf.to_crs(epsg=3857)
gdf_idfm_small_m = gdf_idfm_small.to_crs(epsg=3857)
gdf_idfm_m = gdf_idfm.to_crs(epsg=3857)

for target in ["bus", "metro", "tramway", "train"]:
    target_df = gdf_idfm_small_m[gdf_idfm_small_m[f"nb_{target}_per_day"] > 0]

    nearest = gpd.sjoin_nearest(
        gdf_dvf_m,
        target_df[['stop_id', 'geometry']],
        how='left',
        distance_col='dist_m'
    )

    # ajouter résultats (distance en m et km, id du stop le plus proche) au GeoDataFrame original
    nearest = nearest.reset_index(drop=True)

    gdf_dvf[f"nearest_{target}_stop_id"] = nearest['stop_id']
    gdf_dvf[f"nearest_{target}_dist_m"] = nearest['dist_m']

# afficher le résultat
gdf_dvf[['adresse', 'nearest_bus_stop_id', 'nearest_bus_dist_m', 'nearest_metro_stop_id', 'nearest_metro_dist_m', 'nearest_tramway_stop_id', 'nearest_tramway_dist_m', 'nearest_train_stop_id', 'nearest_train_dist_m']].head()

,adresse,nearest_bus_stop_id,nearest_bus_dist_m,nearest_metro_stop_id,nearest_metro_dist_m,nearest_tramway_stop_id,nearest_tramway_dist_m,nearest_train_stop_id,nearest_train_dist_m
0,1 ALL ADRIENNE,IDFM:73300,155.389997,IDFM:426280,3222.044283,IDFM:73312,708.300133,IDFM:73297,810.600601
1,1 ALL ANDRE MALRAUX,IDFM:74261,264.083932,IDFM:69884,71707.338011,IDFM:68293,52890.886380,IDFM:62168,1451.228854
2,1 ALL ANDRE MALRAUX,IDFM:65153,251.057114,IDFM:71517,22369.191978,IDFM:480927,6986.766072,IDFM:73604,3325.780821
3,1 ALL ANTOINE GROSSIN,IDFM:70505,261.182603,IDFM:70671,2109.592489,IDFM:70310,2226.866100,IDFM:70505,261.182603
4,1 ALL ARAGON,IDFM:73498,154.031537,IDFM:426280,15289.094394,IDFM:73411,7031.585384,IDFM:73482,1142.816999


## Calcul de l'offre de transport dans un rayon de 1km

- **Objectif :** pour chaque point de DVF, compter et agréger l'offre de transport présente dans un rayon de 1 km.
- **Étapes :** 
    - on crée un disque de rayon 1 km autour de chaque logement.
    - on effectue une jointure spatiale pour obtenir tous les couples arrêt situés à ≤ 1 km.
    - dans chauque disque aggrège par ligne de transport (*"route"*) puis par mode (*"route_type"*) pour obtenir le nombre de passage de transport en commun par jour et par mode autour de chaque logement. L'aggrègation d'abord par ligne permet d'éviter de compter plusieurs fois un moyen de transport qui s'arrête plusieurs fois autour d'un logment.
    - après quelques étapes supplémentaires (pivot, renommage, filtrage), les résultats sont ajoutés à DVF

In [2]:
radius_m = 1000 # Distance seuil

# Créer des buffers (disques) de rayon 1 km autour de chaque point DVF
# Utilise gdf_dvf_m (déjà en CRS métrique) pour créer les buffers
gdf_dvf_buffers = gdf_dvf_m.copy()
gdf_dvf_buffers['geometry'] = gdf_dvf_m.geometry.buffer(radius_m)

# Jointure spatiale : trouver tous les couples (point DVF, arrêt de transport) où l'arrêt intersecte le buffer du point
joined = gpd.sjoin(
    gdf_dvf_buffers[['point_id', 'geometry']], 
    gdf_idfm_m[['stop_id', 'route_id', 'nb_stops_per_day', 'route_type', 'geometry']], 
    how='left',  # Jointure gauche pour garder tous les points DVF, même sans intersections
    predicate='intersects' 
).drop(columns=['index_right', 'geometry']).reset_index(drop=True) # Supprimer les colonnes inutiles après jointure


# Agrégation première : par point_id, route_id et route_type, prendre le max de nb_stops_per_day par route
# Cela évite de compter plusieurs fois une route qui passe par plusieurs arrêts dans le rayon
grouped = joined.groupby(
    ['point_id', 'route_id', 'route_type'],
    dropna=False,  # Garder les groupes avec NaN
    as_index=False
).agg(
    nb_stops_per_day_route=('nb_stops_per_day', 'max')  # Max passages par route
).reset_index(drop=True)

# Agrégation seconde : par point_id et route_type, sommer les passages, compter les routes et stations uniques
grouped2 = joined.groupby(['point_id', 'route_type'],
    dropna=False,
    as_index=False).agg(
    passage_journalier=('nb_stops_per_day', 'sum'),  # Somme des passages journaliers par mode
    nb_routes=('route_id', 'nunique'),  # Nombre de routes uniques par mode
    nb_stations=('stop_id', 'nunique')  # Nombre de stations uniques par mode
).reset_index(drop=True)

# Pivot de grouped2 pour avoir une colonne par mode de transport (route_type)
# Les valeurs sont passage_journalier, nb_routes, nb_stations pour chaque mode
grouped2_pivot = grouped2.pivot_table(
    index='point_id',
    columns='route_type', 
    values=['passage_journalier', 'nb_routes', 'nb_stations'],  
    dropna=False,  # Garder les NaN
    fill_value=0  # Remplir les valeurs manquantes par 0 (aucun passage/routes/stations)
)

# Renommer les colonnes pour mapper les codes route_type aux noms de modes
grouped2_pivot = grouped2_pivot.rename(columns={
    0: "tramway",  # 0 -> tramway
    1: "metro",   # 1 -> metro
    2: "train",   # 2 -> train
    3: "bus",     # 3 -> bus
    6: "IGNORED", # 6 -> ignoré (autres modes)
    7: "IGNORED"  # 7 -> ignoré (autres modes)
})

# Aplatir les colonnes multi-index
grouped2_pivot.columns = [
    f"{name}_{mode}_1km"  # Format : passage_journalier_tramway_1km
    for name, mode in grouped2_pivot.columns
]

# Réinitialiser l'index pour avoir point_id comme colonne
grouped2_pivot = grouped2_pivot.reset_index()

# Supprimer les colonnes contenant '_nan_' ou 'IGNORED' (modes non pertinents)
grouped2_pivot = grouped2_pivot.loc[:, ~(grouped2_pivot.columns.str.contains('_nan_') | grouped2_pivot.columns.str.contains('IGNORED'))]

# Fusionner les résultats agrégés dans gdf_dvf pour créer gdf_dvf_final
gdf_dvf_final = gdf_dvf.merge(
    grouped2_pivot,
    on='point_id',  
    how='left'
)

## Visualisation du résultat

Pour illustre les données obtenues, on affiche le premier point de DVF. La carte interactive montre :
- Un marqueur bleu pour le point DVF.
- Des marqueurs rouges pour les arrêts de transport à moins de 1 km (avec passages journaliers agrégés par mode).
- Des marqueurs verts pour les arrêts les plus proches par mode de transport (bus, métro, tramway, train), avec distances en mètres.

In [5]:
from branca.element import Template, MacroElement
from script.leaflet_tools import FondCarteLeaflet
import folium
import great_tables as gt


# affichage des stops within radius on a map for the first point only
df = joined

m = FondCarteLeaflet(afficher_grande_couronne=True).get_map()

first_point = gdf_dvf.iloc[0]
folium.CircleMarker(
    location=[first_point.geometry.y, first_point.geometry.x],
    popup=first_point['point_id'],
    color='blue'
).add_to(m)
for _, row in df[df['point_id'] == first_point['point_id']].iterrows():
    stop = gdf_idfm_small[gdf_idfm_small['stop_id'] == row['stop_id']]

    stop0 = stop.iloc[0]
    route_name = stop0.get('stop_name', None)

    folium.Marker(
        location=[stop0.geometry.y, stop0.geometry.x],
        tooltip=f"Nearby stop:\n{route_name}",
        icon=folium.Icon(color='red', icon='info-sign')
    ).add_to(m)

m

for target in ["bus", "metro", "tramway", "train"]:
    stop_id = first_point.get(f"nearest_{target}_stop_id")
    stop = gdf_idfm_small[gdf_idfm_small['stop_id'] == stop_id]

    stop0 = stop.iloc[0]
    route_name = stop0.get('stop_name', None)

    folium.Marker(
        location=[stop0.geometry.y, stop0.geometry.x],
        tooltip=f"Nearest {target} stop:\n{route_name}",
        icon=folium.Icon(color='green', icon='info-sign')
    ).add_to(m)

# add legend once (do not recreate inside the loop)
legend_html = """
{% macro html(this, kwargs) %}
<div style="position: fixed; 
    bottom: 50px; left: 50px; width: 220px; padding:8px;
    border:2px solid grey; z-index:9999; font-size:14px;
    background-color:white; box-shadow:2px 2px 6px rgba(0,0,0,0.15);
    ">
  <b>Legend</b><br>
  <span style="display:inline-block;width:12px;height:12px;background:blue;border-radius:50%;margin-right:8px;vertical-align:middle;"></span>
    Point (DVF sample)<br>
  <span style="display:inline-block;width:12px;height:12px;background:red;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>
    Nearby stop (within radius)<br>
  <span style="display:inline-block;width:12px;height:12px;background:green;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>
    Nearest stop (by mode)<br>
  <hr style="margin:6px 0"/>
  <small>Hover or click markers for details</small>
</div>
{% endmacro %}
"""
legend = MacroElement()
legend._template = Template(legend_html)
m.get_root().add_child(legend)


# Sauvegarde de la carte 
from pathlib import Path

assets = Path("assets/maps")
assets.mkdir(parents=True, exist_ok=True)

m.save(assets / "main1.html")

[![Interactive map](/assets/maps/main1.png)](/assets/maps/main1.html)

In [6]:
# visualisation de first_point avec great_tables

# visualisation de first_point avec great_tables

df = first_point.reset_index()
df.columns = ['Key', 'Value']
gt.GT(df).tab_header(title="First DVF Point Details", subtitle="The last 9 columns correspond to transportation, the others to housing")


GT(_tbl_data=                               Key                             Value
0                          adresse                    1 ALL ADRIENNE
1                    Date mutation                        02/01/2024
2                  Valeur fonciere                          153000.0
3                      Code_postal                             93250
4                          Commune                       VILLEMOMBLE
5              Surface reelle bati                              37.0
6                  Surface terrain                               0.0
7        Nombre pieces principales                               2.0
8                     Code commune                                77
9                       Type local                       Appartement
10  Valeur foncière au mètre carré                       4135.135135
11                          search  1 ALL ADRIENNE 93250 VILLEMOMBLE
12                    result_score                          0.852845
13                        geometry        POINT (2.506387 48.892543)
14                        point_id                                 0
15             nearest_bus_stop_id                        IDFM:73300
16              nearest_bus_dist_m                        155.389997
17           nearest_metro_stop_id                       IDFM:426280
18            nearest_metro_dist_m                       3222.044283
19         nearest_tramway_stop_id                        IDFM:73312
20          nearest_tramway_dist_m                        708.300133
21           nearest_train_stop_id                        IDFM:73297
22            nearest_train_dist_m                        810.600601, _body=<great_tables._gt_data.Body object at 0x7fc287493e00>, _boxhead=Boxhead([ColInfo(var='Key', type=<ColInfoTypeEnum.default: 1>, column_label='Key', column_align='left', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7fc287493a10>, _spanners=Spanners([]), _heading=Heading(title='First DVF Point Details', subtitle='The last 9 columns correspond to transportation, the others to housing', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7fc2850c42f0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7fc2850bdf90>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7fc2850c4440>, _formats=[], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='b

In [7]:
# ajout d'une colonne departement à gdf_dvf_final

gdf_dvf_final["Department"] = (gdf_dvf_final["Code_postal"].astype(int) // 1000).astype(str)

# enregistrer le GeoDataFrame final en GeoJSON
gdf_dvf_final.to_file("cache/results/logements_transport_final.geojson", driver="GeoJSON", encoding="utf-8")